In [6]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Any, Annotated, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [7]:
# ==== state

class JokeState(TypedDict):
    topic: str
    joke: str 
    explaination: str 
    

In [8]:
def generate_joke(state: JokeState) -> Any: 
    prompt = f'Generate a joke on topic {state['topic']}'
    response = llm.invoke(prompt).content
    return {'joke': response}

def generate_explaination(state: JokeState) -> Any:
    prompt = f'Write an explaination for the joke {state['joke']}'
    response = llm.invoke(prompt)
    return {'explaination': response}

In [9]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [ ]:
config = {'configurable': {'thread_id': 1}}
workflow.invoke({'topic': 'Pilot'}, config=config)

# Get state
workflow.get_state(config)

# Get state history
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'Pilot', 'joke': 'Why did the pilot break up with the runway?\n\nBecause he felt like they were going in circles and he needed more space!', 'explaination': AIMessage(content='Here\'s an explanation of the joke "Why did the pilot break up with the runway? Because he felt like they were going in circles and he needed more space!":\n\nThis joke plays on a **pun** and uses **double meanings** to create humor. Let\'s break it down:\n\n*   **"Why did the pilot break up with the runway?"** This sets up a scenario of a relationship ending, but instead of people, it\'s a pilot and a runway. This is already a bit absurd and hints at wordplay.\n\n*   **"Because he felt like they were going in circles..."**\n    *   **Literal meaning (for a runway):** Runways are where airplanes **taxi and take off**, which often involves moving in a circular pattern on the ground before lining up for departure or after landing. Pilots also perform "circling approaches" for landing

: 